In [ ]:
import numpy as np
import fitsio
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from time import time
import sys

sys.path.append('/global/homes/j/jmccull/sompz/')
from sompz import SelfOrganizingMap
from sompz import *
from sompz.som import *
#kernel imsimv2

# Make a simple SOM for COSMOS and XMM to fill with potential target selections

In [ ]:
class SOM:
    def __init__(self, x, y, input_dim, learning_rate=0.5, sigma=6,random_start=False):
        self.x = x
        self.y = y
        self.input_dim = input_dim
        self.learning_rate = learning_rate
        self.sigma = sigma if sigma is not None else max(x, y) / 2
        if random_start:
            self.weights = np.random.random((x, y, input_dim))
        else:
            self.weights = self._initialize_weights_grid()
        self.history = []
        self.history.append((np.NaN, self.weights.copy()))

    def _initialize_weights_grid(self):
        """Initialize weights in an equidistant grid."""
        # Create a grid of (x, y) coordinates
        xx, yy = np.meshgrid(np.linspace(0, 1, self.x), np.linspace(0, 1, self.y))
        # Flatten the grid coordinates and stack them
        grid = np.stack([xx.ravel(), yy.ravel()], axis=-1)
        # If input_dim > 2, pad the remaining dimensions with zeros
        if self.input_dim > 2:
            grid = np.hstack([grid, np.random.random((self.x * self.y, self.input_dim - 2))])
        # Reshape the grid to (x, y, input_dim)
        return grid.reshape(self.x, self.y, self.input_dim)
        
    @classmethod
    def whiten_data(self,data,mode='linear',verbose=False):
        #will take data and turn it into a 0 to 1 range, and preserve the transformation bounds
        datalims = [np.nanmin(data,axis=0,keepdims=True).T,np.nanmax(data,axis=0,keepdims=True).T]
        if verbose:
            print(datalims[0].shape)
            print(datalims[1].shape)
        assert (datalims[1][:] - datalims[0][:] > 0).all()
        newdata = np.zeros_like(data)
        if mode == 'linear':
            for i in np.arange(data.shape[1]):
                newdata[:,i] = (data[:,i] - datalims[0][i])/(datalims[1][i]- datalims[0][i])

        if mode == 'log':
            for i in np.arange(data.shape[1]):
                newdata[:,i] = (data[:,i] - datalims[0][i])/(datalims[1][i]- datalims[0][i])
        else:
            print("other data modes tbd, for now use 'linear'")
            
            return
        return newdata, datalims
    
    def train(self, data, num_iterations):
        for iteration in range(num_iterations):
            for i, sample in enumerate(data):
                bmu = self._find_bmu(sample)
                self._update_weights(sample, bmu, iteration, num_iterations)
            self.history.append((iteration, self.weights.copy()))
    
    def _find_bmu(self, sample):
        diff = self.weights - sample
        dist = np.linalg.norm(diff, axis=-1)
        return np.unravel_index(np.argmin(dist, axis=None), dist.shape)
    
    @staticmethod
    def find_bmu_custom(weights, sample):
        diff = weights - sample
        dist = np.linalg.norm(diff, axis=-1)
        return np.unravel_index(np.argmin(dist, axis=None), dist.shape)
    
    @staticmethod
    def get_occup(som,weights,samples):
        cells = np.zeros((som.x,som.y))
        for sample in samples:
            idx = SOM.find_bmu_custom(weights,sample)
            cells[idx]=cells[idx]+1
        return cells
    
    def _update_weights(self, sample, bmu, iteration, num_iterations):
        lr = self.learning_rate * (1 - iteration / num_iterations)
        sigma = self.sigma * (1 - iteration / num_iterations)
        bmu_x, bmu_y = bmu
        for x in range(self.x):
            for y in range(self.y):
                dist_to_bmu = np.linalg.norm([x - bmu_x, y - bmu_y])
                if dist_to_bmu <= sigma:
                    influence = np.exp(-dist_to_bmu**2 / (2 * (sigma**2)))
                    self.weights[x, y] += lr * influence * (sample - self.weights[x, y])

In [ ]:
# let's train it on HSC photometry:
cosmos_deep_fname = '/global/cfs/cdirs/desi/users/bid13/DESI_II/target_data/HSC_COSMOS_I_mag_lim_24.8.fits'
xmm_deep_fname = '/global/cfs/cdirs/desi/users/bid13/DESI_II/target_data/HSC_XMM_I_mag_lim_24.8.fits'

In [ ]:
cosmos_deep = fitsio.read(cosmos_deep_fname,ext=1)
print(cosmos_deep.shape)
print(cosmos_deep.dtype.names)

In [ ]:
mask = (cosmos_deep['g_mask_brightstar_any'])&(cosmos_deep['r_mask_brightstar_any'])&(cosmos_deep['i_mask_brightstar_any'])&(cosmos_deep['z_mask_brightstar_any'])&(cosmos_deep['y_mask_brightstar_any'])
#& (cosmos_deep['g_mask_brightstar_halo'])& (cosmos_deep['g_mask_brightstar_dip'])& (cosmos_deep['g_mask_brightstar_ghost'])& (cosmos_deep['g_mask_brightstar_blooming'])
print(np.sum(~mask)/len(mask))
plt.hexbin(cosmos_deep[~mask]['ra'],cosmos_deep[~mask]['dec'])
plt.title('COSMOS')
plt.xlabel('ra')
plt.ylabel('dec')
plt.show()

cosmos_deep = cosmos_deep[mask]

In [ ]:
xmm_deep = fitsio.read(xmm_deep_fname,ext=1)
print(xmm_deep.shape)
print(xmm_deep.dtype.names)
mask = (xmm_deep['g_mask_brightstar_any'])&(xmm_deep['r_mask_brightstar_any'])&(xmm_deep['i_mask_brightstar_any'])&(xmm_deep['z_mask_brightstar_any'])&(xmm_deep['y_mask_brightstar_any'])

print(np.sum(~mask)/len(mask))
plt.hexbin(xmm_deep[~mask]['ra'],xmm_deep[~mask]['dec'])
plt.title('XMM')
plt.xlabel('ra')
plt.ylabel('dec')
plt.show()

xmm_deep = xmm_deep[mask]

# Now find the photometry to train a SOM
We have:
- 

In [ ]:
def abmags_nanojy(fluxes,zp=31.4):
    return zp - 2.5*np.log10(fluxes)
lims = (-1,3,19,28)
plt.hexbin(abmags_nanojy(xmm_deep['g_psfflux_flux']) - abmags_nanojy(xmm_deep['r_psfflux_flux']), abmags_nanojy(xmm_deep['r_psfflux_flux']),extent=lims)
plt.xlabel(r'$g - r$')
plt.ylabel(r'$r$')
plt.show()
plt.hist(abmags_nanojy(xmm_deep['i_psfflux_flux']),bins=np.linspace(19,29,num=100))
plt.xlabel(r'$i_{AB}$')
plt.show()

In [ ]:
bands = ['g','r','i','z','y']
hsc_fluxcols = [band + '_psfflux_flux'for band in bands]
magcols = [band+'_mag' for band in bands]

hsc_fluxerrcols = [band + '_psfflux_fluxerr'for band in bands]
magerr = [band + '_magerr' for band in bands]

hsc_xmm_df = pd.DataFrame(xmm_deep.byteswap().newbyteorder(),columns=xmm_deep.dtype.names)

def get_mag_err(fluxerr,flux):
    snr = flux/fluxerr
    return (2.5/2.3)/snr

for i,band in enumerate(bands):
    print(i,band)
    hsc_xmm_df[magcols[i]] = abmags_nanojy(hsc_xmm_df[hsc_fluxcols[i]].values)
    hsc_xmm_df[magerr[i]] = get_mag_err(hsc_xmm_df[hsc_fluxerrcols[i]].values,hsc_xmm_df[hsc_fluxcols[i]].values)
    hsc_xmm_df[band + 'mag_ivar'] = 1/hsc_xmm_df[magerr[i]]**2
    
print(hsc_xmm_df.columns)

In [ ]:
hsc_xmm_df

In [ ]:
colors = ['purple','blue','green','orange','red']
plt.figure(figsize=(10,5))
for i,band in enumerate(bands):
    plt.hist(hsc_xmm_df[band+'mag_ivar'],alpha=0.5,histtype='step',bins=np.logspace(0,3.5,num=100),linewidth=2.5,color=colors[i],label = 'ivar in ' + band)
plt.xlabel('mag ivar')
plt.legend(loc='upper right')
plt.show()

plt.figure(figsize=(10,5))
for i,band in enumerate(bands):
    plt.hist(hsc_xmm_df[band+'_mag'],alpha=0.5,histtype='step',bins=np.linspace(20,28,num=100),linewidth=2.5,color=colors[i],label = 'mag in ' + band)
plt.xlabel('mag ivar')
plt.legend(loc='upper right')
plt.show()

In [ ]:
cols = [band +'_mag' for band in bands]
ivar_cols = [band+'mag_ivar' for band in bands]

#randomly sample the HSC data:
rand_ind = np.random.randint(0, len(hsc_xmm_df), size=1000, dtype=int).astype(int)


convert_dict = {'g_mag': float, 
                'r_mag': float, 
                'i_mag': float, 
                'z_mag': float, 
                'y_mag': float,
                'gmag_ivar': float, 
                'rmag_ivar': float, 
                'imag_ivar': float, 
                'zmag_ivar': float, 
                'ymag_ivar': float 
               } 
hsc_xmm_df.astype(convert_dict,inplace=True)
somvals = hsc_xmm_df.iloc[rand_ind][cols].values
som_ivar = hsc_xmm_df.iloc[rand_ind][ivar_cols].values
cols = cols+ivar_cols
data = hsc_xmm_df.iloc[rand_ind][cols].values
#sompz is causing problems, let's just do my simple som
som = SOM(x=10, y=10, input_dim=data.shape[1], random_start=True)

#whiten the data
print(data.shape)
data_one,datalims = som.whiten_data(data,mode='linear',verbose=True)
print(data_one)

In [ ]:
start = time() #train the custom som with a small test
som.train(data_one, num_iterations=500)
print('time elapsed: {}'.format(time()-start))    

In [ ]:
#look at the occupancy for a simple 10 by 10 som

#at the start:
it,weights = som.history[0]
#print(weights)
occup = som.get_occup(som,weights,data)
plt.imshow(occup.reshape(10,10))
plt.title('before training')
plt.show()

it,weights = som.history[1]
#print(weights)
occup = som.get_occup(som,weights,data)
plt.imshow(occup.reshape(10,10))
plt.title('middle training')
plt.show()

#at the end
it,weights = som.history[-1]
occup = som.get_occup(som,weights,data)

plt.imshow(occup.reshape(10,10))
plt.title('after training')
plt.show()

In [ ]:
som

In [ ]:
#som = train_som(somvals, som_ivar, map_shape=[10, 10], learning_rate=0.5, max_iter=1e3, min_val=1e-4, verbose=True, diag_ivar=False, replace=False)
"""Calculate Self Organizing Map

Parameters
----------
x :             input data of shape (n_samples, n_dim)
ivar :          inverse variance of input data (n_samples, n_dim, n_dim)
map_shape :     desired output map shape = [dim1, dim2]. (n_out_dim,)
learning_rate : float usually between 0 and 1. Sets how large of a
                change we can effect in the weights at each step by
                multiplying the change by:
                    learning_rate ** (step_t / total_t)
max_iter :      maximum number of steps in algorithm fit
min_val :       minimum parameter difference we worry about in updating
                SOM. This in practice usually doesn't come up, as we limit
                the range of cells a SOM may update to be less than one
                wrap around the map.
verbose :       Print updates?

Returns
-------
w : self organizing map weights (dim1, dim2, n_dims)

Notes
-----
Suggest whitening your data to span 0-1 range before training

"""